# LensForge: Gravitational Lensing Research Notebook

This interactive notebook demonstrates the local LensForge physics and visualization engine:
1. **Point-Mass Gravitational Lensing**: Relativistic Einstein radius, thin-screen ray-tracing, analytical point-image solutions, parity, and magnifications.
2. **Realistic Galaxy Lensing**: Singular Isothermal Sphere (SIS), Singular Isothermal Ellipsoid (SIE, Kormann et al. 1994), External Gravitational Shear, and giant gravitational arcs.


## 1. Imports & Package Verification
All calculations run natively on your local workstation.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

import lensforge
from lensforge.constants import M_SUN, MEGAPARSEC, arcsec2rad, rad2arcsec
from lensforge.physics.coordinates import make_coordinate_grid
from lensforge.physics.distances import (
    einstein_radius,
    einstein_radius_arcsec,
    einstein_radius_sis,
    einstein_radius_sis_arcsec,
)
from lensforge.physics.models import (
    PointMassLens,
    SingularIsothermalSphere,
    SingularIsothermalEllipsoid,
    ExternalShear,
    CompositeLens,
)
from lensforge.imaging.source_models import GaussianSource
from lensforge.visualization.lens_plots import plot_lensing_system, plot_galaxy_lensing_system

# Ensure local outputs directory exists
output_dir = Path("../outputs").resolve()
output_dir.mkdir(parents=True, exist_ok=True)

print(f"LensForge Version: {lensforge.__version__}")
print(f"Local outputs directory: {output_dir}")

## 2. Point-Mass Gravitational Lensing
Compute the relativistic Einstein radius for a $10^{11} M_\odot$ lens at $D_l = 1000\text{ Mpc}$ and background source at $D_s = 1800\text{ Mpc}$.

In [ ]:
# 1. Physical cosmological setup
mass_kg = 1.0e11 * M_SUN
d_l_m = 1000.0 * MEGAPARSEC
d_s_m = 1800.0 * MEGAPARSEC
d_ls_m = 1100.0 * MEGAPARSEC

theta_e_rad = einstein_radius(mass_kg, d_l_m, d_s_m, d_ls_m)
theta_e_arcsec = einstein_radius_arcsec(mass_kg, d_l_m, d_s_m, d_ls_m)
print(f"Einstein Radius: {theta_e_rad:.6e} rad = {theta_e_arcsec:.4f} arcsec")

# 2. Instantiate point-mass lens model
lens = PointMassLens(theta_e=theta_e_arcsec, center=(0.0, 0.0))

# 3. Background source galaxy
src_center = (0.28, 0.18)
source = GaussianSource.from_degrees(
    center=src_center,
    sigma=0.12,
    amp=1.0,
    axis_ratio=0.8,
    pos_angle_deg=35.0,
)

# 4. Image-plane grid & backwards ray-tracing
grid = make_coordinate_grid(num_pix=400, field_of_view=4.0, unit="arcsec")
unlensed = source.evaluate(grid.x, grid.y)
bx, by = lens.ray_trace(grid.x, grid.y, mask_singular=True)
lensed = np.nan_to_num(source.evaluate(bx, by), nan=0.0)

# 5. Analytical image positions and magnification
img_info = lens.image_positions_2d(*src_center)
mag_info = lens.image_magnifications_2d(*src_center)
print(f"Image positions: {img_info['images']}")
print(f"Signed magnifications: mu_+ = {mag_info['mu_plus']:.4f}, mu_- = {mag_info['mu_minus']:.4f}")
print(f"Invariant verification (mu_+ + mu_-): {mag_info['mu_plus'] + mag_info['mu_minus']:.6f} == 1.0")

# 6. Render local plot
fig, _ = plot_lensing_system(
    grid=grid,
    unlensed_image=unlensed,
    lensed_image=lensed,
    theta_e=theta_e_arcsec,
    lens_center=(0.0, 0.0),
    source_center=src_center,
    image_positions=img_info['images'],
    output_path=str(output_dir / "notebook_point_mass.png"),
)
plt.show()

## 3. Singular Isothermal Sphere (SIS)
Demonstrate the SIS galaxy model parameterized by 1D velocity dispersion $\sigma_v$.

In [ ]:
# Galaxy velocity dispersion sigma_v = 240 km/s
sigma_v = 240_000.0  # m/s
theta_e_sis_arcsec = einstein_radius_sis_arcsec(sigma_v, d_s_m, d_ls_m)
print(f"SIS Einstein Radius for sigma_v = 240 km/s: {theta_e_sis_arcsec:.4f} arcsec")

sis_lens = SingularIsothermalSphere(theta_e=theta_e_sis_arcsec, center=(0.0, 0.0))
t_plus, t_minus = sis_lens.scalar_image_positions(0.3)
print(f"Scalar image positions: theta_+ = {t_plus:.4f}, theta_- = {t_minus:.4f}")
print(f"Separation invariant (theta_+ - theta_-): {t_plus - t_minus:.4f} == {2*theta_e_sis_arcsec:.4f}")

## 4. Realistic Galaxy Lensing: SIE + External Shear
Simulate a massive elliptical galaxy ($q = 0.72, \phi = 45^\circ$) with tidal external shear ($\gamma = 0.06, \phi_\gamma = 20^\circ$), generating giant gravitational arcs.

In [ ]:
# 1. SIE Halo + External Shear Composite Model
sie = SingularIsothermalEllipsoid.from_degrees(
    theta_e=1.35,
    axis_ratio=0.72,
    pos_angle_deg=45.0,
    center=(0.0, 0.0),
)
shear = ExternalShear.from_polar_degrees(
    gamma=0.06,
    phi_gamma_deg=20.0,
    center=(0.0, 0.0),
)
composite = CompositeLens([sie, shear])

# 2. Offset background source galaxy
galaxy_src = GaussianSource.from_degrees(
    center=(0.18, 0.12),
    sigma=0.10,
    amp=1.0,
    axis_ratio=0.80,
    pos_angle_deg=30.0,
)

# 3. Ray-tracing and mass convergence
grid_gal = make_coordinate_grid(num_pix=450, field_of_view=4.5, unit="arcsec")
unlensed_gal = galaxy_src.evaluate(grid_gal.x, grid_gal.y)
bx_gal, by_gal = composite.ray_trace(grid_gal.x, grid_gal.y, mask_singular=True)
lensed_gal = np.nan_to_num(galaxy_src.evaluate(bx_gal, by_gal), nan=0.0)
kappa_map = sie.convergence_analytic(grid_gal.x, grid_gal.y, mask_singular=True)

# 4. Render 3-panel visualization
fig, _ = plot_galaxy_lensing_system(
    grid=grid_gal,
    unlensed_image=unlensed_gal,
    lensed_image=lensed_gal,
    convergence_map=kappa_map,
    lens_center=(0.0, 0.0),
    source_center=(0.18, 0.12),
    output_path=str(output_dir / "notebook_galaxy_sie.png"),
    title="Singular Isothermal Ellipsoid (SIE) + External Shear",
)
plt.show()
print(f"Diagnostic plot saved to: {output_dir / 'notebook_galaxy_sie.png'}")